In [9]:
import pandas as pd

In [10]:
nb = pd.read_csv("neighborhood.csv")

In [11]:
nb.describe()

,packet_count,total_bytes,min_rssi,max_rssi,avg_rssi,mgmt_pkts,ctrl_pkts,data_pkts
count,3.180239e+06,3.180239e+06,3.180239e+06,3.180239e+06,3.180239e+06,3.180239e+06,3.180239e+06,3.180239e+06
mean,2.017748e+01,5.305107e+03,-6.593352e+01,-6.213622e+01,-6.382140e+01,1.640432e+01,2.192014e+00,1.581149e+00
std,3.976804e+01,7.196968e+03,1.036680e+01,1.171932e+01,1.105893e+01,1.673668e+01,3.407050e+01,6.555091e+00
min,1.000000e+00,3.400000e+01,-8.800000e+01,-8.600000e+01,-8.600000e+01,0.000000e+00,0.000000e+00,0.000000e+00
25%,2.000000e+00,5.130000e+02,-7.600000e+01,-7.400000e+01,-7.420000e+01,2.000000e+00,0.000000e+00,0.000000e+00
50%,1.300000e+01,3.792000e+03,-6.800000e+01,-6.400000e+01,-6.500000e+01,1.200000e+01,0.000000e+00,0.000000e+00
75%,3.100000e+01,8.616000e+03,-5.600000e+01,-5.000000e+01,-5.200000e+01,2.800000e+01,0.000000e+00,1.000000e+00
max,3.083000e+03,1.430062e+06,-1.800000e+01,-1.600000e+01,-1.800000e+01,3.083000e+03,2.804000e+03,1.227000e+03


In [12]:
nb.info()

<class 'pandas.DataFrame'>
RangeIndex: 3180239 entries, 0 to 3180238
Data columns (total 11 columns):
 #   Column             Dtype  
---  ------             -----  
 0   window_timestamp   str    
 1   mac                str    
 2   packet_count       int64  
 3   total_bytes        int64  
 4   min_rssi           int64  
 5   max_rssi           int64  
 6   avg_rssi           float64
 7   mgmt_pkts          int64  
 8   ctrl_pkts          int64  
 9   data_pkts          int64  
 10  observed_channels  str    
dtypes: float64(1), int64(7), str(3)
memory usage: 266.9 MB


In [13]:
from collections import Counter

# Comprehensive human-readable initial exploration of `nb`
# (executes in-place using existing `nb` DataFrame)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

def explore_nb(df, n_sample=10):
    print("== Basic info ==")
    print(f"Rows: {df.shape[0]:,}    Columns: {df.shape[1]}\n")

    print("== Columns, dtypes, non-null counts, unique counts, % missing ==")
    info_df = pd.DataFrame({
        "dtype": df.dtypes,
        "non_null": df.notna().sum(),
        "unique": df.nunique(dropna=False),
        "missing_%": (df.isna().mean() * 100).round(3)
    })
    print(info_df)
    print()

    # Memory usage
    try:
        mem = df.memory_usage(deep=True).sum()
        print(f"Estimated memory usage: {mem / (1024**2):.2f} MB\n")
    except Exception:
        pass

    print("== First and last rows (sample) ==")
    display(df.head(n_sample))
    display(df.tail(n_sample))
    print()

    # Timestamp parsing and time range
    ts_col = 'window_timestamp'
    if ts_col in df.columns:
        df[ts_col] = pd.to_datetime(df[ts_col], errors='coerce')
        ts_min = df[ts_col].min()
        ts_max = df[ts_col].max()
        print("== Timestamp range ==")
        print(f"Min: {ts_min}    Max: {ts_max}    Span: {ts_max - ts_min}")
        print("Rows with invalid/missing timestamps:", df[ts_col].isna().sum())
        print()
        # coverage by day (top 10)
        print("Top 10 days by record count:")
        print(df[ts_col].dt.date.value_counts().head(10))
        print()

    # Univariate numeric summaries with extended percentiles
    num_cols = df.select_dtypes(include=['number']).columns.tolist()
    if num_cols:
        print("== Numeric summary (extended percentiles) ==")
        desc = df[num_cols].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T
        print(desc[['count','mean','std','min','1%','5%','25%','50%','75%','95%','99%','max']])
        print()
        # quick zeros / negatives for key cols
        for c in num_cols:
            zeros = (df[c] == 0).sum()
            neg = (df[c] < 0).sum()
            if zeros or neg:
                print(f"{c}: zeros={zeros:,}, negatives={neg:,}")
        print()

    # RSSI-specific notes
    rssi_cols = [c for c in ['min_rssi','max_rssi','avg_rssi'] if c in df.columns]
    if rssi_cols:
        print("== RSSI distribution (counts by rounded dBm) ==")
        for c in rssi_cols:
            print(f"-- {c} top values --")
            print(df[c].round().value_counts().head(10))
        print()

    # Categorical / identifier columns
    id_cols = ['mac']
    print("== Identifier / categorical columns ==")
    if 'mac' in df.columns:
        print(f"Unique MAC addresses: {df['mac'].nunique():,}")
        print("Top 10 MACs by row count:")
        print(df['mac'].value_counts().head(10))
        print()

    # Aggregate per MAC (top by total packets/bytes)
    agg_cols = []
    if 'packet_count' in df.columns:
        agg_cols.append('packet_count')
    if 'total_bytes' in df.columns:
        agg_cols.append('total_bytes')
    if agg_cols:
        print("== Top 10 MACs by aggregate traffic ==")
        agg = df.groupby('mac')[agg_cols].sum().sort_values(by=agg_cols[0], ascending=False).head(10)
        display(agg)
        print()

    # observed_channels parsing
    if 'observed_channels' in df.columns:
        print("== observed_channels overview ==")
        sample_obs = df['observed_channels'].dropna().astype(str).head(10).tolist()
        print("Sample observed_channels values:", sample_obs)
        # quick stats: presence of multiple channels
        has_pipe = df['observed_channels'].astype(str).str.contains(r'\|', na=False).sum()
        unique_obs = df['observed_channels'].nunique(dropna=True)
        print(f"Unique observed_channels values: {unique_obs}")
        print(f"Rows with multiple channels (contains '|'): {has_pipe:,}")
        # explode channel tokens for frequency (careful with memory: do for top tokens only)
        ch_series = df['observed_channels'].dropna().astype(str).str.split(r'\|')
        # create value counts of channel tokens using iteration to avoid large intermediate if needed
        cnt = Counter()
        for lst in ch_series.head(200000):  # limit iteration to first 200k rows for speed/safety
            cnt.update(lst)
        top_tokens = cnt.most_common(20)
        print("Top observed channel tokens (from first 200k rows):", top_tokens)
        print()

    # Correlations (numeric)
    if len(num_cols) > 1:
        print("== Correlations (pearson) among numeric columns (top 10 pairs by abs corr) ==")
        corr = df[num_cols].corr().abs().unstack().dropna()
        corr = corr[corr < 1].sort_values(ascending=False).drop_duplicates()
        print(corr.head(10))
        print()

    # Missingness patterns (top columns with missing)
    miss = (df.isna().mean() * 100).sort_values(ascending=False)
    print("== Columns with missing values (percent) ==")
    print(miss[miss > 0].head(20))
    print()

    # Quick anomalies / checks
    print("== Quick anomaly checks ==")
    if 'packet_count' in df.columns:
        high_packet_rows = df[df['packet_count'] > df['packet_count'].quantile(0.999)].shape[0]
        print(f"Rows with packet_count > 99.9th percentile: {high_packet_rows:,}")
    if 'total_bytes' in df.columns:
        zero_bytes_rows = df[(df['packet_count']>0) & (df['total_bytes']==0)].shape[0]
        print(f"Rows with packet_count>0 but total_bytes==0: {zero_bytes_rows:,}")
    print()

    print("Exploration complete. Adjust sample sizes/limits in function for heavier or lighter processing.")

# Run exploration (will modify window_timestamp to datetime in-place)
explore_nb(nb)

== Basic info ==
Rows: 3,180,239    Columns: 11

== Columns, dtypes, non-null counts, unique counts, % missing ==
                     dtype  non_null  unique  missing_%
window_timestamp       str   3180237  557550        0.0
mac                    str   3180239   78899        0.0
packet_count         int64   3180239    1666        0.0
total_bytes          int64   3180239   33965        0.0
min_rssi             int64   3180239      36        0.0
max_rssi             int64   3180239      36        0.0
avg_rssi           float64   3180239     556        0.0
mgmt_pkts            int64   3180239     160        0.0
ctrl_pkts            int64   3180239    1600        0.0
data_pkts            int64   3180239     456        0.0
observed_channels      str   3180239      11        0.0

Estimated memory usage: 771.63 MB

== First and last rows (sample) ==


,window_timestamp,mac,packet_count,total_bytes,min_rssi,max_rssi,avg_rssi,mgmt_pkts,ctrl_pkts,data_pkts,observed_channels
0,2026-05-27T13:46:09.586928,2ccea7194345943d,59,19408,-78,-44,-48.0,55,0,4,1
1,2026-05-27T13:46:09.586928,348c047dc0d24307,39,9171,-50,-44,-47.5,39,0,0,1
2,2026-05-27T13:46:09.586928,d8ae690aea494c19,8,2584,-76,-74,-75.0,8,0,0,1
3,2026-05-27T13:46:09.586928,6264afbcc1b180b7,1,44,-66,-66,-66.0,0,0,1,1
4,2026-05-27T13:46:09.586928,72ef1975aafb2148,7,1392,-78,-74,-75.7,7,0,0,1|6
5,2026-05-27T13:46:09.586928,a6c30b44b9d3ceb8,40,10357,-56,-52,-54.0,33,0,7,6
6,2026-05-27T13:46:09.586928,dfb8c33c743b71c9,20,7014,-76,-72,-74.4,17,1,2,11
7,2026-05-27T13:46:09.586928,4a885d11fbc3b0b3,19,6517,-76,-74,-74.9,19,0,0,11
8,2026-05-27T13:46:19.602387,a6c30b44b9d3ceb8,31,11910,-56,-52,-54.1,26,0,5,6
9,2026-05-27T13:46:19.602387,dfb8c33c743b71c9,46,15179,-80,-72,-74.6,34,0,12,11


,window_timestamp,mac,packet_count,total_bytes,min_rssi,max_rssi,avg_rssi,mgmt_pkts,ctrl_pkts,data_pkts,observed_channels
3180229,2026-08-17T10:10:40.413279,2ccea7194345943d,32,9194,-54,-52,-53.8,30,0,2,6
3180230,2026-08-17T10:10:40.413279,348c047dc0d24307,22,5214,-54,-52,-53.6,22,0,0,6
3180231,2026-08-17T10:10:40.413279,f1ac77e4c83ddf4d,2,92,-70,-70,-70.0,0,2,0,11
3180232,2026-08-17T10:10:50.423628,72ef1975aafb2148,1,220,-72,-72,-72.0,1,0,0,6
3180233,2026-08-17T10:10:50.423628,348c047dc0d24307,15,3555,-54,-54,-54.0,15,0,0,6
3180234,2026-08-17T10:10:50.423628,2ccea7194345943d,17,4964,-54,-52,-53.4,17,0,0,6
3180235,2026-08-17T10:10:50.423628,a6c30b44b9d3ceb8,57,12021,-64,-48,-53.4,36,19,2,11
3180236,2026-08-17T10:10:50.423628,6264afbcc1b180b7,5,198,-64,-62,-62.8,0,2,3,11
3180237,2026-08-17T10:10:50.423628,f1ac77e4c83ddf4d,4,136,-74,-72,-72.5,0,4,0,11
3180238,2026-08-17T10:10:50.423628,b115a68ac07ff598,1,274,-72,-72,-72.0,1,0,0,1



== Timestamp range ==
Min: 2026-05-27 13:46:09.586928    Max: 2026-08-17 10:10:50.423628    Span: 81 days 20:24:40.836700
Rows with invalid/missing timestamps: 6

Top 10 days by record count:
window_timestamp
2026-07-23    80315
2026-07-22    77665
2026-07-21    65090
2026-06-14    63415
2026-06-13    62518
2026-06-03    62457
2026-06-15    62337
2026-06-12    61164
2026-06-05    60252
2026-06-30    59603
Name: count, dtype: int64

== Numeric summary (extended percentiles) ==
                  count         mean          std   min    1%    5%    25%     50%     75%      95%      99%        max
packet_count  3180239.0    20.177480    39.768036   1.0   1.0   1.0    2.0    13.0    31.0     53.0     80.0     3083.0
total_bytes   3180239.0  5305.106890  7196.968497  34.0  44.0  88.0  513.0  3792.0  8616.0  15216.0  22220.0  1430062.0
min_rssi      3180239.0   -65.933523    10.366796 -88.0 -80.0 -78.0  -76.0   -68.0   -56.0    -50.0    -48.0      -18.0
max_rssi      3180239.0   -62.136221  

,packet_count,total_bytes
mac,,
a6c30b44b9d3ceb8,21495846,5563601986
2ccea7194345943d,15542639,4508426157
348c047dc0d24307,11656948,2748710290
dfb8c33c743b71c9,2425560,820955425
4a885d11fbc3b0b3,1692099,580299163
b115a68ac07ff598,1583111,406263720
d8ae690aea494c19,1557671,463282224
29674906ccec77de,1248780,406444598
d003bf67eaa76aa3,1133685,310895483



== observed_channels overview ==
Sample observed_channels values: ['1', '1', '1', '1', '1|6', '6', '11', '11', '6', '11']
Unique observed_channels values: 11
Rows with multiple channels (contains '|'): 82,028
Top observed channel tokens (from first 200k rows): [('1', 91478), ('11', 60431), ('6', 52991)]

== Correlations (pearson) among numeric columns (top 10 pairs by abs corr) ==
avg_rssi     max_rssi        0.988968
min_rssi     avg_rssi        0.956139
max_rssi     min_rssi        0.930339
ctrl_pkts    packet_count    0.882363
total_bytes  mgmt_pkts       0.700750
mgmt_pkts    max_rssi        0.673608
             avg_rssi        0.645996
total_bytes  packet_count    0.605316
mgmt_pkts    min_rssi        0.574946
total_bytes  data_pkts       0.544382
dtype: float64

== Columns with missing values (percent) ==
window_timestamp    0.000189
dtype: float64

== Quick anomaly checks ==
Rows with packet_count > 99.9th percentile: 3,175
Rows with packet_count>0 but total_bytes==0: 0

Explo

In [14]:
# inspect invalid/missing and out-of-range timestamps
ts = 'window_timestamp'
if ts not in nb.columns:
    print("No window_timestamp column found")
else:
    total = len(nb)
    invalid_mask = nb[ts].isna()
    n_invalid = int(invalid_mask.sum())
    print(f"Invalid/missing timestamps: {n_invalid} / {total} ({n_invalid/total:.3%})")

    if n_invalid:
        print("\nSample rows with invalid timestamps (first 20):")
        display(nb.loc[invalid_mask].head(20))

    # define a plausible range and flag out-of-range timestamps
    min_ok = pd.Timestamp('2000-01-01')
    max_ok = pd.Timestamp.now() + pd.Timedelta(days=1)
    oor_mask = nb[ts].notna() & ((nb[ts] < min_ok) | (nb[ts] > max_ok))
    n_oor = int(oor_mask.sum())
    print(f"\nOut-of-range timestamps (<{min_ok.date()} or >{max_ok.date()}): {n_oor}")

    if n_oor:
        print("\nSample out-of-range rows (first 20):")
        display(nb.loc[oor_mask, [ts,'mac','packet_count','observed_channels']].head(20))

    # show range of valid timestamps
    valid_ts = nb[ts].dropna()
    if not valid_ts.empty:
        print(f"\nValid timestamp range: {valid_ts.min()} -> {valid_ts.max()}")
    else:
        print("\nNo valid timestamps present.")

Invalid/missing timestamps: 6 / 3180239 (0.000%)

Sample rows with invalid timestamps (first 20):


,window_timestamp,mac,packet_count,total_bytes,min_rssi,max_rssi,avg_rssi,mgmt_pkts,ctrl_pkts,data_pkts,observed_channels
75548,NaT,348c047dc0d24307,53,12465,-48,-46,-47.3,53,0,0,1
1331139,NaT,a6c30b44b9d3ceb8,40,11265,-62,-52,-59.2,34,0,6,11
1331140,NaT,dfc1a1af46d57570,2,84,-72,-70,-71.0,0,0,2,1
1331141,NaT,dfb8c33c743b71c9,2,340,-76,-76,-76.0,0,1,1,6
1331142,NaT,4a885d11fbc3b0b3,1,343,-76,-76,-76.0,1,0,0,6
2907842,NaT,a6c30b44b9d3ceb8,81,12018,-64,-48,-53.4,32,42,7,1



Out-of-range timestamps (<2000-01-01 or >2026-08-18): 0

Valid timestamp range: 2026-05-27 13:46:09.586928 -> 2026-08-17 10:10:50.423628
